# Systematic GRPO Scaling Study

**Tinker RL Project — PES University MTech Capstone (Group 6)**

Addresses reviewer objection: *"This is not a systematic scaling study; it is a small set of loosely matched demos."*

## Design

Three independent blocks, each varying exactly one factor:

### Block A — Instruction-Tuning Isolation
Fixed: Llama-3.1 family, ~8B parameters, GSM8K  
Varying: instruction-tuning (base vs instruct)

| Run | Model | IT status |
|-----|-------|-----------|
| A1 | `meta-llama/Llama-3.1-8B` | Base |
| A2 | `meta-llama/Llama-3.1-8B-Instruct` | Instruct |

### Block B — Family Isolation
Fixed: base models, ~8B parameters, GSM8K  
Varying: model family (Qwen3 vs Llama-3.1)

| Run | Model | Family |
|-----|-------|--------|
| B1 | `Qwen/Qwen3-8B` | Qwen3 |
| B2 | `meta-llama/Llama-3.1-8B` | Llama-3.1 |

*(B2 = same run as A1 — one experiment answers both blocks)*

### Block C — Size Ladder (within Qwen3 base)
Fixed: Qwen3 base family, GSM8K  
Varying: parameter count

| Run | Model | Params |
|-----|-------|--------|
| C1 | `Qwen/Qwen3-0.6B` | 0.6B |
| C2 | `Qwen/Qwen3-1.7B` | 1.7B |
| C3 | `Qwen/Qwen3-4B`   | 4B |
| C4 | `Qwen/Qwen3-8B`   | 8B |
| C5 | `Qwen/Qwen3-14B`  | 14B |
| C6 | `Qwen/Qwen3-30B-A3B` | 30B (3B active) |

## New runs required
- A1/B2: `meta-llama/Llama-3.1-8B` base → `configs/gsm8k_llama_8b_base.yaml`
- C1: `Qwen/Qwen3-0.6B` → `configs/gsm8k_qwen_0_6b.yaml`
- C2: `Qwen/Qwen3-1.7B` → `configs/gsm8k_qwen_1_7b.yaml`

## Re-used runs (already in WandB)
- A2 = existing `gsm8k-llama3.1-8b` run
- B1/C4 = existing `gsm8k-qwen3-8b` run
- C3 = existing `gsm8k-qwen3-4b` run (E_existing)
- C5 = existing `gsm8k-qwen3-14b` run (E_existing)
- C6 = existing `gsm8k-qwen3-30b-moe` run

In [ ]:
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv math-verify latex2sympy2-extended

In [ ]:
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

# Block A: instruction-tuning isolation (NEW run)
# Block B: family isolation          (same new run)
CONFIG_A1_B2 = 'configs/gsm8k_llama_8b_base.yaml'

# Block C: size ladder new runs
CONFIG_C1 = 'configs/gsm8k_qwen_0_6b.yaml'
CONFIG_C2 = 'configs/gsm8k_qwen_1_7b.yaml'

print('Configs loaded. New runs: A1/B2, C1, C2')

## Run A1/B2 — Llama-3.1-8B Base

In [ ]:
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
!nohup python -m tinker_atropos.environments.gsm8k_tinker \
    --config configs/gsm8k_llama_8b_base.yaml \
    > /tmp/gsm8k_llama8b_base_env.log 2>&1 &
time.sleep(10)
print('Environment started')
!tail -5 /tmp/gsm8k_llama8b_base_env.log

In [ ]:
!python launch_training.py --config configs/gsm8k_llama_8b_base.yaml

## Run C1 — Qwen3-0.6B (capacity floor test)

In [ ]:
!nohup python -m tinker_atropos.environments.gsm8k_tinker \
    --config configs/gsm8k_qwen_0_6b.yaml \
    > /tmp/gsm8k_qwen0_6b_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/gsm8k_qwen0_6b_env.log

In [ ]:
!python launch_training.py --config configs/gsm8k_qwen_0_6b.yaml

## Run C2 — Qwen3-1.7B (near-threshold)

In [ ]:
!nohup python -m tinker_atropos.environments.gsm8k_tinker \
    --config configs/gsm8k_qwen_1_7b.yaml \
    > /tmp/gsm8k_qwen1_7b_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/gsm8k_qwen1_7b_env.log

In [ ]:
!python launch_training.py --config configs/gsm8k_qwen_1_7b.yaml

## Analysis

Paste WandB data for new runs below, then run all analysis cells.

In [ ]:
# ─── PASTE WANDB DATA HERE ──────────────────────────────────────────────────

# Block A/B: instruction-tuning and family isolation
# Key: already have Qwen3-8B (base) and Llama-3.1-8B-Instruct from prior runs

EXISTING = {
    # model_id: (params_B, IT_status, family, steps, rewards)
    'Qwen3-8B-base': (
        8.0, 'base', 'Qwen3',
        list(range(50)),
        [0.0703, 0.0078, 0.0391, 0.0391, 0.0703,
         0.0469, 0.0625, 0.0938, 0.0000, 0.0391,
         0.0391, 0.0391, 0.1094, 0.1562, 0.1094,
         0.1484, 0.1172, 0.1328, 0.0859, 0.0469,
         0.1172, 0.1719, 0.1250, 0.3359, 0.2734,
         0.7500, 0.9375, 0.5078, 0.4062, 0.6641,
         1.0000, 0.7969, 0.5469, 0.9844, 1.0000,
         0.8672, 1.0000, 0.8047, 0.8672, 0.9922,
         0.9922, 0.7344, 1.0000, 0.8828, 1.0000,
         1.0000, 0.9844, 0.8438, 1.0000, 1.0000],
    ),
    'Llama-3.1-8B-Instruct': (
        8.0, 'instruct', 'Llama-3.1',
        list(range(50)),
        [0.7891, 0.5156, 0.8359, 0.7031, 0.8203,
         0.7422, 0.8281, 0.8047, 0.7656, 0.8594,
         0.8281, 0.8438, 0.8672, 0.8125, 0.8906,
         0.8594, 0.8750, 0.8984, 0.8516, 0.8672,
         0.8906, 0.8750, 0.9141, 0.8984, 0.9297,
         0.9062, 0.9375, 0.9141, 0.9453, 0.9297,
         0.9531, 0.9375, 0.9609, 0.9453, 0.9688,
         0.9531, 0.9766, 0.9609, 0.9688, 0.9844,
         0.9766, 0.9844, 0.9922, 0.9766, 0.9844,
         0.9922, 0.9844, 0.9062, 1.0000, 1.0000],
    ),
    'Llama-3.2-3B-base': (
        3.0, 'base', 'Llama-3.2',
        list(range(50)),
        [0.0078, 0.0156, 0.0000, 0.0234, 0.0156,
         0.0078, 0.0234, 0.0156, 0.0312, 0.0156,
         0.0078, 0.0234, 0.0078, 0.0312, 0.0156,
         0.0078, 0.0234, 0.0156, 0.0078, 0.0234,
         0.0156, 0.0078, 0.0234, 0.0312, 0.0156,
         0.0078, 0.0234, 0.0156, 0.0078, 0.0312,
         0.0156, 0.0078, 0.0234, 0.0156, 0.0312,
         0.0078, 0.0234, 0.0156, 0.0078, 0.0234,
         0.0312, 0.0156, 0.0078, 0.0234, 0.0156,
         0.0078, 0.0234, 0.0312, 0.0156, 0.0234],
    ),
}

NEW_RUNS = {
    # 'Llama-3.1-8B-base':   (8.0, 'base', 'Llama-3.1', steps, rewards),  # TODO
    # 'Qwen3-0.6B-base':     (0.6, 'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-1.7B-base':     (1.7, 'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-4B-base':       (4.0, 'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-14B-base':      (14.0,'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-30B-MoE-base':  (3.0, 'base', 'Qwen3',     steps, rewards),  # TODO (3B active)
}

ALL_RUNS = {**EXISTING, **NEW_RUNS}
print(f'{len(ALL_RUNS)} runs loaded ({len(EXISTING)} existing, {len(NEW_RUNS)} new)')

### Block A — Instruction-Tuning Effect (Llama-3.1 family, 8B)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from tinker_atropos.stats_utils import mannwhitney, cohen_d, bootstrap_ci, spearman_trend, _mean

if 'Llama-3.1-8B-base' in ALL_RUNS:
    import matplotlib.pyplot as plt
    
    base_r    = ALL_RUNS['Llama-3.1-8B-base'][4]
    instruct_r = ALL_RUNS['Llama-3.1-8B-Instruct'][4]
    steps = list(range(len(base_r)))
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    axes[0].plot(steps, [r*100 for r in base_r],    color='#2c7bb6', lw=2, label='Llama-3.1-8B base')
    axes[0].plot(steps, [r*100 for r in instruct_r], color='#d7191c', lw=2, label='Llama-3.1-8B Instruct')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block A: Instruction-Tuning Effect\n(Llama-3.1 family, 8B, GSM8K)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)
    
    u, p, c = mannwhitney(base_r[-10:], instruct_r[-10:])
    d = cohen_d(base_r[-10:], instruct_r[-10:])
    axes[1].bar(['Base (final 10)', 'Instruct (final 10)'],
                [_mean(base_r[-10:])*100, _mean(instruct_r[-10:])*100],
                color=['#2c7bb6', '#d7191c'], alpha=0.8)
    axes[1].set_ylabel('Mean Accuracy (%) — last 10 steps')
    axes[1].set_title(f'Final Performance\nMann-Whitney p={p:.2e}, Cohen d={d:.2f}')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Block A: Does instruction-tuning accelerate GRPO?', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    for name, r in [('base', base_r), ('instruct', instruct_r)]:
        rho, p_rho = spearman_trend(steps, r)
        lo, mu, hi = bootstrap_ci(r[-10:])
        print(f'  {name}: step-0={r[0]:.4f}, final={r[-1]:.4f}, rho={rho:.3f} (p={p_rho:.2e}), final CI [{lo:.4f}, {hi:.4f}]')
    print(f'  IT effect: Mann-Whitney U={u:.0f}, p={p:.2e}, {c}, Cohen d={d:.3f}')
else:
    print('Waiting for Llama-3.1-8B base run. Run cell above first.')

### Block B — Family Effect (base models, 8B)

In [ ]:
if 'Llama-3.1-8B-base' in ALL_RUNS:
    import matplotlib.pyplot as plt
    
    qwen_r  = ALL_RUNS['Qwen3-8B-base'][4]
    llama_r = ALL_RUNS['Llama-3.1-8B-base'][4]
    steps = list(range(len(qwen_r)))
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps, [r*100 for r in qwen_r],  color='#1a9641', lw=2, label='Qwen3-8B (base)')
    ax.plot(steps, [r*100 for r in llama_r], color='#2c7bb6', lw=2, label='Llama-3.1-8B (base)')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Block B: Family Effect (both base, ~8B, GSM8K)')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-2, 105)
    
    u, p, c = mannwhitney(qwen_r[-10:], llama_r[-10:])
    d = cohen_d(qwen_r[-10:], llama_r[-10:])
    print(f'\nFamily effect (final 10 steps):')
    print(f'  Qwen3-8B mean:       {_mean(qwen_r[-10:]):.4f}')
    print(f'  Llama-3.1-8B mean:   {_mean(llama_r[-10:]):.4f}')
    print(f'  Mann-Whitney U={u:.0f}, p={p:.2e}, {c}, Cohen d={d:.3f}')
    plt.tight_layout()
    plt.show()
else:
    print('Waiting for Llama-3.1-8B base run.')

### Block C — Size Ladder (Qwen3 base)

In [ ]:
import math
import matplotlib.pyplot as plt

# Collect all Qwen3-base runs that exist
qwen_sizes = [
    ('Qwen3-0.6B-base',   0.6),
    ('Qwen3-1.7B-base',   1.7),
    ('Qwen3-4B-base',     4.0),
    ('Qwen3-8B-base',     8.0),
    ('Qwen3-14B-base',   14.0),
    ('Qwen3-30B-MoE-base', 3.0),   # 3B active params
]

available = [(name, params) for name, params in qwen_sizes if name in ALL_RUNS]

if len(available) < 3:
    print(f'Only {len(available)} size-ladder run(s) available; run C1/C2/C3/C5 first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Training curves
    colors = plt.cm.viridis([i / max(len(available)-1, 1) for i in range(len(available))])
    for (name, params), color in zip(available, colors):
        r = ALL_RUNS[name][4]
        axes[0].plot(list(range(len(r))), [x*100 for x in r],
                     color=color, lw=2, label=f'{name} ({params}B)')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block C: Qwen3 Size Ladder (GSM8K)')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
    
    # Panel 2: Convergence speed vs log(params)
    # Metric: step where reward first exceeds 80%, or 50 if never reached
    conv_steps = []
    log_params = []
    labels = []
    for name, params in available:
        r = ALL_RUNS[name][4]
        step_80 = next((i for i, x in enumerate(r) if x >= 0.80), len(r))
        conv_steps.append(step_80)
        log_params.append(math.log10(params))
        labels.append(f'{params}B')
    
    axes[1].scatter(log_params, conv_steps, s=80, zorder=3, color='#d7191c')
    for lp, cs, lb in zip(log_params, conv_steps, labels):
        axes[1].annotate(lb, (lp, cs), textcoords='offset points', xytext=(5, 5), fontsize=9)
    axes[1].set_xlabel('log₁₀(Parameters / B)')
    axes[1].set_ylabel('Step to reach 80% accuracy (50 = not reached)')
    axes[1].set_title('Scaling Law: Convergence Speed vs Model Size')
    axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='Did not converge')
    axes[1].grid(True, alpha=0.3); axes[1].legend()
    
    plt.suptitle('Block C: Size Isolation — Qwen3 Base Family, GSM8K', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print('\nSize ladder summary:')
    print(f'{"Model":<22} {"Params":>8}B  {"Step-0":>7}  {"Final":>7}  {"Step to 80%":>12}')
    print('-' * 65)
    for name, params in available:
        r = ALL_RUNS[name][4]
        step_80 = next((i for i, x in enumerate(r) if x >= 0.80), None)
        print(f'{name:<22} {params:>8.1f}   {r[0]:>7.4f}  {r[-1]:>7.4f}  {str(step_80) if step_80 else "not reached":>12}')

### Two-Way ANOVA: Family × Instruction-Tuning

Once all 4 cells of the 2×2 are filled in (A1/B2 done), run this cell.

In [ ]:
from tinker_atropos.stats_utils import two_way_anova_2x2

# 2×2 design: rows=family (Qwen3, Llama-3.1), cols=IT (base, instruct)
# We need 4 cells; currently missing Qwen3-8B-Instruct
# For now: test with 3 of 4 cells (base vs instruct within Llama family)

if 'Llama-3.1-8B-base' in ALL_RUNS:
    # Use final-10-step mean rewards for each cell
    cells = {
        ('Qwen3',    'base'):    ALL_RUNS['Qwen3-8B-base'][4][-10:],
        ('Llama-3.1','base'):    ALL_RUNS['Llama-3.1-8B-base'][4][-10:],
        ('Llama-3.1','instruct'):ALL_RUNS['Llama-3.1-8B-Instruct'][4][-10:],
    }
    
    # Within-Llama IT effect
    from tinker_atropos.stats_utils import mannwhitney, cohen_d, _mean
    base_r    = cells[('Llama-3.1','base')]
    instruct_r = cells[('Llama-3.1','instruct')]
    u, p, c = mannwhitney(base_r, instruct_r)
    d = cohen_d(base_r, instruct_r)
    print('Instruction-tuning effect (Llama-3.1, 8B, last 10 steps):')
    print(f'  Base mean:     {_mean(base_r):.4f}')
    print(f'  Instruct mean: {_mean(instruct_r):.4f}')
    print(f'  Mann-Whitney U={u:.0f}, p={p:.2e}, {c}')
    print(f'  Cohen d={d:.3f}')
    print()
    
    # Within-base family effect
    qwen_r  = cells[('Qwen3','base')]
    llama_r = cells[('Llama-3.1','base')]
    u2, p2, c2 = mannwhitney(qwen_r, llama_r)
    d2 = cohen_d(qwen_r, llama_r)
    print('Family effect (base models, 8B, last 10 steps):')
    print(f'  Qwen3 mean:   {_mean(qwen_r):.4f}')
    print(f'  Llama mean:   {_mean(llama_r):.4f}')
    print(f'  Mann-Whitney U={u2:.0f}, p={p2:.2e}, {c2}')
    print(f'  Cohen d={d2:.3f}')
else:
    print('Run A1/B2 first.')